In [ ]:
install.packages("tidyverse")
install.packages ("magrittr")
install.packages("pheatmap")
library(tidyverse)
library(magrittr)
library(pheatmap)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘magrittr’


The following object is masked from ‘package:purrr’:

    set_names


The following object is masked from ‘package:tidyr’:

    extract




In [ ]:
raw_data <- read.csv('../data/final_merged_set.repair_measurements.csv')

In [ ]:
renamed_data <- raw_data %>%
  mutate(sample = str_replace(sample, "U2OS_OGG1", "U2OSPatty_OGG1"))

In [ ]:
format_data <- function(input_df){
  # Reformat the data a little
  input_df <- input_df %>%
  rename(DRC = "value") %>%
  separate(
    col = sample,
    into = c("cell_line", "KO_gene","time_point","replicate"),
    sep = "_",
    remove = FALSE) %>%
  mutate(time_point = parse_number(time_point)) %>%
  select(sample, cell_line, KO_gene, time_point, replicate, pathway, DRC) %>%
# Create sample_id that includes cell_line, KO_bene, and timpe_point
    mutate(sample_brief = paste(paste(cell_line, KO_gene, time_point, sep = "_"),"h", sep = ''))
return(input_df)
}

In [ ]:
formatted_data <- renamed_data %>% format_data

In [ ]:
normalize_drc <- function(input_df) {
  logit_pathways <- c("BER_8oxoG_C", "BER_A_8oxoG", "BER_Hx", "BER_UG", "DR_O6MeG", "MMR")
  log_pathways   <- c("NER", "NHEJ", "MMEJ")

  # Define the fixed constant
  offset <- 1e-5

  # Create a temporary column to protect boundaries before math happens
  input_df$DRC_adj <- input_df$DRC
  input_df$DRC_adj[input_df$DRC_adj == 0] <- offset # replace 0 with 1e-5 to avoid -inf after log transformation
  input_df$DRC_adj[input_df$DRC_adj == 1] <- 1 - offset # replace 1 with 1-(1e-5) to avoid a denominator of 0 in logit

  # Apply masks for the specific pathways
  mask1 <- input_df$pathway %in% logit_pathways
  mask2 <- input_df$pathway %in% log_pathways

  # Overwrite original DRC column using the adjusted values
  input_df$DRC[mask1] <- log(input_df$DRC_adj[mask1] / (1 - input_df$DRC_adj[mask1]))
  input_df$DRC[mask2] <- log(input_df$DRC_adj[mask2])

  # Clean up the temporary column
  input_df$DRC_adj <- NULL

  return(input_df)
}

In [ ]:
normalized_data <- formatted_data %>% normalize_drc

In [ ]:
normalized_data

sample,cell_line,KO_gene,time_point,replicate,pathway,DRC,sample_brief
<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>
U2OS_XPG_4h_BR3,U2OS,XPG,4,BR3,BER_8oxoG_C,2.9522472,U2OS_XPG_4h
U2OS_XPG_4h_BR3,U2OS,XPG,4,BR3,BER_A_8oxoG,0.8932990,U2OS_XPG_4h
U2OS_XPG_4h_BR3,U2OS,XPG,4,BR3,BER_Hx,0.4761199,U2OS_XPG_4h
U2OS_XPG_4h_BR3,U2OS,XPG,4,BR3,BER_UG,6.2479979,U2OS_XPG_4h
U2OS_XPG_4h_BR3,U2OS,XPG,4,BR3,DR_O6MeG,0.8420395,U2OS_XPG_4h
U2OS_XPG_4h_BR3,U2OS,XPG,4,BR3,MMEJ,-3.3583075,U2OS_XPG_4h
U2OS_XPG_4h_BR3,U2OS,XPG,4,BR3,MMR,-1.9844060,U2OS_XPG_4h
U2OS_XPG_4h_BR3,U2OS,XPG,4,BR3,NER,-4.7073305,U2OS_XPG_4h
U2OS_XPG_4h_BR3,U2OS,XPG,4,BR3,NHEJ,1.3165895,U2OS_XPG_4h


In [ ]:
get_DRC_replicates <- function(input_df, pathway_value){
  v <- input_df %>%
    filter(pathway == pathway_value) %>%
         pull (DRC)
  return(v)
}

In [ ]:
create_joint_df <- function(input_df1, input_df2){
  # for the given ko cell line (input_df_1), create a joint df containing the name of the ko sample and its corresponding wt sample (input_df_2)
  df_joint <- input_df1 %>% inner_join(input_df2, by = "pathway", suffix = c("_1", "_2")) %>%
    select(sample_brief_1 = sample_brief_1,
    sample_brief_2 = sample_brief_2,
    pathway)
  return(df_joint)
}

In [ ]:
get_t_statistics <- function(input_df, sample_brief_ko, sample_brief_wt){
  #input_df should be formatted
  #Given the name of the KO sample and the WT sample, get the joint_df with t-statistics
  ko_df <- input_df %>% filter(sample_brief == sample_brief_ko)
  wt_df <- input_df %>% filter(sample_brief == sample_brief_wt)
  ##create the joint df
  df_joint <- create_joint_df(ko_df, wt_df) %>% unique
  ##retrieve pathway list
  pathway_list <- df_joint$pathway
  ##create a vector to store the t-statistics and p-value
  t_stats_list <- c()
  p_value_list <- c()
  ## get p-value and t-statistic for each pathway
  for (pw in pathway_list){
    #get the DRC vector for the ko and the DRC vector for the corresponding wild type.
    ko_vector <- get_DRC_replicates(ko_df, pw)
    wt_vector <- get_DRC_replicates(wt_df, pw)
    #Perform t-test. Store the t-statistic and the p-value in back to the joint_df
    t_test_result_welch <- t.test(ko_vector, wt_vector, paired = FALSE)

    t_stat <- t_test_result_welch$statistic %>% as.numeric()
    t_stats_list <- c(t_stats_list, t_stat)

    p_value <- t_test_result_welch$p.value %>% as.numeric()
    p_value_list <- c(p_value_list, p_value)
  }
  ## store the vectors as columns in joint df
  df_joint %>%
     mutate(t_stats = t_stats_list) %>%
     mutate(p_value = p_value_list)
}

In [ ]:
KO_list <- c(
    'U2OS_MSH2_4h',
    'U2OS_MGMT_2h',
    'U2OSPatty_OGG1_2h',
    'HAP1_MUTYH_2h',
    "U2OS_UNG_2h",
    'HAP1_MPG_2h',
    'U2OS_XPG_4h',
    'TK6_LIG4_2h',
    'TK6_PolQ_4h'
  )

WT_list <- c(
  'U2OS_WT_4h',
  'U2OS_WT_2h',
  'U2OSPatty_WT_2h',
  'HAP1_WT_2h',
  'U2OS_WT_2h',
  'HAP1_WT_2h',
  'U2OS_WT_4h',
  'TK6_WT_2h',
  'TK6_WT_4h'
)

In [ ]:
# create an empty initial data frame
df_t_test <- data.frame(
  sample_brief_1 = character(),
  sample_brief_2 = character(),
  pathway = character(),
  t_stats = double(),
  p_value = double()
)
#iterate through the provided KO_list
n <- length(KO_list)
indices <- 1:n
for (i in indices){
  sample_brief_ko <- KO_list[i]
  sample_brief_wt <- WT_list[i]
  sub_df <- get_t_statistics(normalized_data, sample_brief_ko, sample_brief_wt)
  df_t_test <- rbind(df_t_test, sub_df)
}


Warning message in inner_join(., input_df2, by = "pathway", suffix = c("_1", "_2")):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message in inner_join(., input_df2, by = "pathway", suffix = c("_1", "_2")):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message in inner_join(., input_df2, by = "pathway", suffix = c("_1", "_2")):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expect

In [ ]:
# p-value matrix
pvalue_mat <- df_t_test %>%
  select(sample_brief_1, pathway, p_value) %>%
# Convert to matrix format using pivot_wider
    pivot_wider(
      names_from = pathway,
      values_from = p_value,
      values_fill = NA  # Fill missing values with NA
    )  %>%
    select(sample_brief_1, MMR, DR_O6MeG, BER_8oxoG_C, BER_A_8oxoG, BER_UG, BER_Hx, NER, NHEJ, MMEJ) %>%
    as.data.frame() %>%               # Ensure it's a standard data frame (tibbles don't support row names)
  `rownames<-`(.$sample_brief_1) %>% # Set row names using the sample_brief_1 column
  select(-sample_brief_1) %>%       # Remove the column since it's now the row names
  as.matrix()                       # Convert the remaining pure-numeric data into a true matrix

In [ ]:
pvalue_mat %>% format(scientific = FALSE)

,MMR,DR_O6MeG,BER_8oxoG_C,BER_A_8oxoG,BER_UG,BER_Hx,NER,NHEJ,MMEJ
U2OS_MSH2_4h,0.00054243420,0.00726143294,0.42602902815,0.01576446388,0.94442232175,0.30266708061,0.03958056295,0.02729950847,0.13024494722
U2OS_MGMT_2h,0.57018947892,0.00088362415,0.05066414956,0.52748120423,0.39080180723,0.40479209914,0.10440645415,0.65688591164,0.25593983016
U2OSPatty_OGG1_2h,0.56012297721,0.74829175131,0.01267253230,0.04595999304,0.90715989915,0.87735882356,0.02921745035,0.39238388654,0.68001176977
HAP1_MUTYH_2h,0.09536089387,0.03738711747,0.08048755552,0.00002170272,0.06042535270,0.75787729102,0.00711073092,0.65034558924,0.19487179509
U2OS_UNG_2h,0.00193064711,0.67251050945,0.62278696741,0.47876409701,0.00052454223,0.29157149563,0.04570396636,0.00219893145,0.13445824454
HAP1_MPG_2h,0.66310233015,0.65422656892,0.48323559359,0.22470506906,0.33775523558,0.00157563410,0.00957935283,0.85426007175,0.20283860818
U2OS_XPG_4h,0.19697328712,0.00645295867,0.40958453665,0.33375708844,0.27336430191,0.16836599874,0.00407681156,0.28697188535,0.18790901368
TK6_LIG4_2h,0.46396079683,0.62151546922,0.55456247862,0.65519846447,0.42109278954,0.84074239743,0.01827719689,0.00003135339,0.38283041220
TK6_PolQ_4h,0.96333250412,0.85381011122,0.22639693045,0.91449168013,0.34540769060,0.12541404398,0.07736276634,0.59766313385,0.38558434857


In [ ]:
# after data transformation
# p-value before adjustment
pvalue_mat_2_decimal <- format(round(pvalue_mat,2),
    nsmall = 2,
    scientific = FALSE,
    trim = TRUE)

In [ ]:
pvalue_mat_2_decimal

,MMR,DR_O6MeG,BER_8oxoG_C,BER_A_8oxoG,BER_UG,BER_Hx,NER,NHEJ,MMEJ
U2OS_MSH2_4h,0.00,0.01,0.43,0.02,0.94,0.30,0.04,0.03,0.13
U2OS_MGMT_2h,0.57,0.00,0.05,0.53,0.39,0.40,0.10,0.66,0.26
U2OSPatty_OGG1_2h,0.56,0.75,0.01,0.05,0.91,0.88,0.03,0.39,0.68
HAP1_MUTYH_2h,0.10,0.04,0.08,0.00,0.06,0.76,0.01,0.65,0.19
U2OS_UNG_2h,0.00,0.67,0.62,0.48,0.00,0.29,0.05,0.00,0.13
HAP1_MPG_2h,0.66,0.65,0.48,0.22,0.34,0.00,0.01,0.85,0.20
U2OS_XPG_4h,0.20,0.01,0.41,0.33,0.27,0.17,0.00,0.29,0.19
TK6_LIG4_2h,0.46,0.62,0.55,0.66,0.42,0.84,0.02,0.00,0.38
TK6_PolQ_4h,0.96,0.85,0.23,0.91,0.35,0.13,0.08,0.60,0.39


In [ ]:
fdr_bh <- function(p_mat) {
  # 1. Create a logical mask for the off-diagonal elements
  off_diag_mask <- row(p_mat) != col(p_mat)

  # 2. Apply BH adjustment only to the off-diagonal elements
  p_mat[off_diag_mask] <- p.adjust(p_mat[off_diag_mask], method = "BH")

  return(p_mat)
}

In [ ]:
# after data transformation
# adjust off-diagonal p-value (BH)
# diagonal p-values remained unadjusted
qvalue_mat <- pvalue_mat %>% fdr_bh

In [ ]:
qvalue_mat %>% format(scientific = FALSE)

,MMR,DR_O6MeG,BER_8oxoG_C,BER_A_8oxoG,BER_UG,BER_Hx,NER,NHEJ,MMEJ
U2OS_MSH2_4h,0.00054243420,0.10456463428,0.68164644504,0.16214877134,0.95772404459,0.62262942297,0.23636567851,0.21036564253,0.42091276550
U2OS_MGMT_2h,0.78949312466,0.00088362415,0.24318791787,0.77507442254,0.68164644504,0.68164644504,0.37586323494,0.78969108747,0.59444089585
U2OSPatty_OGG1_2h,0.78949312466,0.85261195240,0.01267253230,0.23636567851,0.94062001385,0.92896816613,0.21036564253,0.68164644504,0.78969108747
HAP1_MUTYH_2h,0.36136759784,0.23636567851,0.32195022208,0.00002170272,0.27191408715,0.85261195240,0.10456463428,0.78969108747,0.52158499246
U2OS_UNG_2h,0.07916153237,0.78969108747,0.78969108747,0.72485339038,0.00052454223,0.61744552016,0.23636567851,0.07916153237,0.42091276550
HAP1_MPG_2h,0.78969108747,0.78969108747,0.72485339038,0.54335263307,0.65445667692,0.00157563410,0.11495223396,0.91801082338,0.52158499246
U2OS_XPG_4h,0.52158499246,0.10456463428,0.68164644504,0.65445667692,0.61506967930,0.50509799621,0.00407681156,0.61744552016,0.52158499246
TK6_LIG4_2h,0.72485339038,0.78969108747,0.78949312466,0.78969108747,0.68164644504,0.91801082338,0.16449477199,0.00003135339,0.68164644504
TK6_PolQ_4h,0.96333250412,0.91801082338,0.54335263307,0.94062001385,0.65445667692,0.42091276550,0.32195022208,0.78969108747,0.38558434857


In [ ]:
qvalue_mat_2_decimal <- format(round(qvalue_mat,2),
    nsmall = 2,
    scientific = FALSE,
    trim = TRUE)

In [ ]:
qvalue_mat_2_decimal

,MMR,DR_O6MeG,BER_8oxoG_C,BER_A_8oxoG,BER_UG,BER_Hx,NER,NHEJ,MMEJ
U2OS_MSH2_4h,0.00,0.10,0.68,0.16,0.96,0.62,0.24,0.21,0.42
U2OS_MGMT_2h,0.79,0.00,0.24,0.78,0.68,0.68,0.38,0.79,0.59
U2OSPatty_OGG1_2h,0.79,0.85,0.01,0.24,0.94,0.93,0.21,0.68,0.79
HAP1_MUTYH_2h,0.36,0.24,0.32,0.00,0.27,0.85,0.10,0.79,0.52
U2OS_UNG_2h,0.08,0.79,0.79,0.72,0.00,0.62,0.24,0.08,0.42
HAP1_MPG_2h,0.79,0.79,0.72,0.54,0.65,0.00,0.11,0.92,0.52
U2OS_XPG_4h,0.52,0.10,0.68,0.65,0.62,0.51,0.00,0.62,0.52
TK6_LIG4_2h,0.72,0.79,0.79,0.79,0.68,0.92,0.16,0.00,0.68
TK6_PolQ_4h,0.96,0.92,0.54,0.94,0.65,0.42,0.32,0.79,0.39
